# Lab type: review
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: LoRA and Parameter-Efficient Fine-tuning
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
# Install dependencies (uncomment if running on Colab)
# !pip install transformers peft torch

from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

print('PEFT version:', __import__('peft').__version__)

## Part 1: LoRA Configuration — Rank and Target Modules

Three LoRA configurations are applied to the same base model. Run the cell and examine the trainable parameter counts.

In [ ]:
def make_peft_model(r, target_modules, base_model_name='distilbert-base-uncased'):
    base = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=4)
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=r,             # unit scaling: alpha == r
        target_modules=target_modules,
        lora_dropout=0.1,
        bias='none',
    )
    peft_model = get_peft_model(base, config)
    trainable, total = peft_model.get_nb_trainable_parameters()
    return peft_model, trainable, total

configs = [
    ('Baseline (no LoRA)',    None,  None,                 None),
    ('r=4,  q only',         4,     ['q_lin'],             None),
    ('r=16, q+v',            16,    ['q_lin', 'v_lin'],    None),
    ('r=64, q+v+k+out',      64,    ['q_lin', 'v_lin', 'k_lin', 'out_lin'], None),
]

# Baseline — all params trainable
baseline = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)
baseline_params = sum(p.numel() for p in baseline.parameters())
print(f'{'Config':<30} {'Trainable':>12} {'Total':>12} {'%':>8}')
print('-' * 66)
print(f'{'Full fine-tuning':<30} {baseline_params:>12,} {baseline_params:>12,} {100.0:>7.2f}%')

for label, r, target_modules, _ in configs[1:]:
    model, trainable, total = make_peft_model(r, target_modules)
    pct = 100 * trainable / total
    print(f'{label:<30} {trainable:>12,} {total:>12,} {pct:>7.2f}%')

**Question 1:** The table shows trainable parameter counts for four configurations. A colleague is fine-tuning for news topic classification (AG News: 4 classes, semantically close to BERT/DistilBERT's web pretraining). They argue that `r=64, q+v+k+out` gives the 'most capacity' and should always outperform `r=16, q+v`. Evaluate this argument — when does higher rank help, and when does it hurt?

*(Write your answer here.)*

**Question 2:** The `lora_alpha` parameter is set equal to `r` in all configurations above (unit scaling). LoRA scales the adapter output by `alpha / r`. If you double `r` but keep `alpha` fixed at 16, how does the effective scaling of the adapter contribution change? What practical effect does this have on early training dynamics?

*(Write your answer here.)*

## Part 2: Target Module Choice

LoRA is applied only to the modules listed in `target_modules`. Run the cell to see which modules are available and which are typically targeted.

In [ ]:
# Inspect DistilBERT attention module names
base = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

print('Attention-related module names in DistilBERT:')
for name, module in base.named_modules():
    if any(attn in name for attn in ['q_lin', 'k_lin', 'v_lin', 'out_lin']):
        param_count = sum(p.numel() for p in module.parameters())
        print(f'  {name:<55} params: {param_count:,}')

In [ ]:
# Compare: query-only vs query+value LoRA for sentiment-adjacent classification
# (Simulated: we examine trainable param counts, not actual task accuracy)
model_q_only, tr_q, tot = make_peft_model(r=16, target_modules=['q_lin'])
model_qv,     tr_qv, _  = make_peft_model(r=16, target_modules=['q_lin', 'v_lin'])

print(f'q only  — trainable: {tr_q:,}')
print(f'q + v   — trainable: {tr_qv:,}')
print(f'Extra trainable params from adding v: {tr_qv - tr_q:,}')
print()
print('Rule of thumb: always include value projection for classification tasks.')
print('Value projections carry the content that gets weighted by attention — this is where')
print('task-specific adaptation signal flows most directly.')

**Question 3:** An AI-generated LoRA config for a medical NER task uses `target_modules=['q_lin']` (query only). You are reviewing it. Give two reasons why this choice is suboptimal for a classification/NER task, and state what you would change and why.

*(Write your answer here.)*

## Part 3: Verifying LoRA Freezes the Base Model

A key property of LoRA is that base model weights are frozen during training — only the adapter matrices `A` and `B` receive gradients.

In [ ]:
# Verify that base weights are frozen and adapter weights are trainable
peft_model, _, _ = make_peft_model(r=16, target_modules=['q_lin', 'v_lin'])

print(f'{'Parameter name':<60} {'Requires grad':>14}')
print('-' * 76)
for name, param in peft_model.named_parameters():
    if 'lora' in name or ('q_lin' in name and 'lora' not in name) or 'classifier' in name:
        print(f'{name:<60} {str(param.requires_grad):>14}')

**Question 4:** The output shows that base weight matrices (`q_lin.weight`) have `requires_grad=False`, while LoRA adapter weights (`lora_A`, `lora_B`) have `requires_grad=True`. If you save and reload the PEFT model, you only need to save the adapter weights, not the full base model. In what two scenarios is this especially valuable in a production deployment?

*(Write your answer here.)*

## Summary

> **Complete each sentence.**

1. For a task semantically close to pretraining (e.g., news classification), the appropriate LoRA rank is _____ because _____.
2. Including only `query` in `target_modules` is typically worse than `query + value` because _____.
3. QLoRA extends LoRA by _____, which reduces memory from _____ to roughly _____ for a 7B model.